# Ripe Pumpkins Recommendation Service

#### Source: 
https://www.codementor.io/@jadianes/building-a-recommender-with-apache-spark-python-example-app-part1-du1083qbw

### Create a SparkContext configured for local mode

In [1]:
import pyspark
sc = pyspark.SparkContext('local[*]')

### File download
#### Small: 100,000 ratings and 3,600 tag applications applied to 9,000 movies by 600 users. Last updated 9/2018.
#### Full: approximately 33,000,000 ratings and 2,000,000 tag applications applied to 86,000 movies by 330,975 users. Includes tag genome data with 14 million relevance scores across 1,100 tags. Last updated 9/2018.

In [2]:
complete_dataset_url = 'http://files.grouplens.org/datasets/movielens/ml-latest.zip'
small_dataset_url = 'http://files.grouplens.org/datasets/movielens/ml-latest-small.zip'

### Download location(s)

In [3]:
import os
datasets_path = os.path.join('/home/jovyan','work')
complete_dataset_path = os.path.join(datasets_path, 'ml-latest.zip')
small_dataset_path = os.path.join(datasets_path, 'ml-latest-small.zip')

### Geting file(s)

In [ ]:
import urllib

small_f = urllib.request.urlretrieve (small_dataset_url, small_dataset_path)
complete_f = urllib.request.urlretrieve (complete_dataset_url, complete_dataset_path)

### Extracting file(s)

In [ ]:
import zipfile

with zipfile.ZipFile(small_dataset_path, "r") as z:
    z.extractall(datasets_path)

with zipfile.ZipFile(complete_dataset_path, "r") as z:
    z.extractall(datasets_path)

### Loading and parsing datasets
#### filter out header of ratings.csv

In [4]:
small_ratings_file = os.path.join(datasets_path, 'ml-latest-small', 'ratings.csv')

small_ratings_raw_data = sc.textFile(small_ratings_file)
small_ratings_raw_data_header = small_ratings_raw_data.take(1)[0]

#### parse the raw data into a new RDD

In [5]:
small_ratings_data = small_ratings_raw_data.filter(lambda line: line!=small_ratings_raw_data_header)\
    .map(lambda line: line.split(",")).map(lambda tokens: (tokens[0],tokens[1],tokens[2])).cache()


In [6]:
small_ratings_data.take(3)

[('1', '1', '4.0'), ('1', '3', '4.0'), ('1', '6', '4.0')]

In [7]:
[(u'1', u'6', u'2.0'), (u'1', u'22', u'3.0'), (u'1', u'32', u'2.0')]


[('1', '6', '2.0'), ('1', '22', '3.0'), ('1', '32', '2.0')]

#### do the same to movies.csv

In [8]:
small_movies_file = os.path.join(datasets_path, 'ml-latest-small', 'movies.csv')

small_movies_raw_data = sc.textFile(small_movies_file)
small_movies_raw_data_header = small_movies_raw_data.take(1)[0]

small_movies_data = small_movies_raw_data.filter(lambda line: line!=small_movies_raw_data_header)\
    .map(lambda line: line.split(",")).map(lambda tokens: (tokens[0],tokens[1])).cache()
    
small_movies_data.take(3)


[('1', 'Toy Story (1995)'),
 ('2', 'Jumanji (1995)'),
 ('3', 'Grumpier Old Men (1995)')]

In [9]:
 [(u'1', u'Toy Story (1995)'),
 (u'2', u'Jumanji (1995)'),
 (u'3', u'Grumpier Old Men (1995)')]


[('1', 'Toy Story (1995)'),
 ('2', 'Jumanji (1995)'),
 ('3', 'Grumpier Old Men (1995)')]

### Collaborative filtering

#### selecting ALS parameters using the small dataset

In [10]:
training_RDD, validation_RDD, test_RDD = small_ratings_data.randomSplit([6, 2, 2], seed=0)

validation_for_predict_RDD = validation_RDD.map(lambda x: (x[0], x[1]))
test_for_predict_RDD = test_RDD.map(lambda x: (x[0], x[1]))



In [11]:
from pyspark.mllib.recommendation import ALS
import math

seed = 5  # Removed the L suffix for Python 3
iterations = 10
regularization_parameter = 0.1
ranks = [4, 8, 12]
errors = [0, 0, 0]
err = 0
tolerance = 0.02

min_error = float('inf')
best_rank = -1
best_iteration = -1

for rank in ranks:
    model = ALS.train(training_RDD, rank, seed=seed, iterations=iterations,
                      lambda_=regularization_parameter)
    predictions = model.predictAll(validation_for_predict_RDD).map(lambda r: ((r[0], r[1]), r[2]))
    rates_and_preds = validation_RDD.map(lambda r: ((int(r[0]), int(r[1])), float(r[2]))).join(predictions)
    error = math.sqrt(rates_and_preds.map(lambda r: (r[1][0] - r[1][1])**2).mean())
    errors[err] = error
    err += 1
    print(f'For rank {rank} the RMSE is {error}')
    if error < min_error:
        min_error = error
        best_rank = rank

print(f'The best model was trained with rank {best_rank}')


For rank 4 the RMSE is 0.908078105265682
For rank 8 the RMSE is 0.916462973348527
For rank 12 the RMSE is 0.917665030756129
The best model was trained with rank 4


In [12]:
predictions.take(3)

[((372, 1084), 3.42419871162954),
 ((4, 1084), 3.866749726695713),
 ((402, 1084), 3.4099577968422152)]

In [13]:
rates_and_preds.take(3)

[((1, 457), (5.0, 4.381060760461434)),
 ((1, 1025), (5.0, 4.705295366590298)),
 ((1, 1089), (5.0, 4.979982471805129))]

In [12]:
model = ALS.train(training_RDD, best_rank, seed=seed, iterations=iterations,
                  lambda_=regularization_parameter)

predictions = model.predictAll(test_for_predict_RDD).map(lambda r: ((r[0], r[1]), r[2]))

rates_and_preds = test_RDD.map(lambda r: ((int(r[0]), int(r[1])), float(r[2]))).join(predictions)

error = math.sqrt(rates_and_preds.map(lambda r: (r[1][0] - r[1][1])**2).mean())

print(f'For testing data the RMSE is {error}')


For testing data the RMSE is 0.9113780946334407


### using the complete dataset to build the final model

In [13]:
complete_ratings_file = os.path.join(datasets_path, 'ml-latest', 'ratings.csv')
complete_ratings_raw_data = sc.textFile(complete_ratings_file)
complete_ratings_raw_data_header = complete_ratings_raw_data.take(1)[0]

complete_ratings_data = complete_ratings_raw_data.filter(lambda line: line != complete_ratings_raw_data_header) \
    .map(lambda line: line.split(",")) \
    .map(lambda tokens: (int(tokens[0]), int(tokens[1]), float(tokens[2]))).cache()


In [14]:
print(f"There are {complete_ratings_data.count()} recommendations in the complete dataset")

There are 33832162 recommendations in the complete dataset


In [16]:
training_RDD, test_RDD = complete_ratings_data.randomSplit([7, 3], seed=0)

complete_model = ALS.train(training_RDD, best_rank, seed=seed, 
                           iterations=iterations, lambda_=regularization_parameter)


In [17]:
test_for_predict_RDD = test_RDD.map(lambda x: (x[0], x[1]))

predictions = complete_model.predictAll(test_for_predict_RDD).map(lambda r: ((r[0], r[1]), r[2]))
rates_and_preds = test_RDD.map(lambda r: ((int(r[0]), int(r[1])), float(r[2]))).join(predictions)
error = math.sqrt(rates_and_preds.map(lambda r: (r[1][0] - r[1][1])**2).mean())

print(f'For testing data the RMSE is {error}')

For testing data the RMSE is 0.8257054095972955


### load the movies

In [18]:
complete_movies_file = os.path.join(datasets_path, 'ml-latest', 'movies.csv')
complete_movies_raw_data = sc.textFile(complete_movies_file)
complete_movies_raw_data_header = complete_movies_raw_data.take(1)[0]

# Parse
complete_movies_data = complete_movies_raw_data.filter(lambda line: line != complete_movies_raw_data_header) \
    .map(lambda line: line.split(",")) \
    .map(lambda tokens: (int(tokens[0]), tokens[1], tokens[2])).cache()

complete_movies_titles = complete_movies_data.map(lambda x: (int(x[0]), x[1]))

print(f"There are {complete_movies_titles.count()} movies in the complete dataset")


There are 86537 movies in the complete dataset


In [19]:
def get_counts_and_averages(ID_and_ratings_tuple):
    nratings = len(ID_and_ratings_tuple[1])
    avg_rating = float(sum(ID_and_ratings_tuple[1])) / nratings
    return ID_and_ratings_tuple[0], (nratings, avg_rating)

movie_ID_with_ratings_RDD = complete_ratings_data.map(lambda x: (x[1], x[2])).groupByKey()
movie_ID_with_avg_ratings_RDD = movie_ID_with_ratings_RDD.map(get_counts_and_averages)
movie_rating_counts_RDD = movie_ID_with_avg_ratings_RDD.map(lambda x: (x[0], x[1][0]))


### adding new user ratings

In [21]:
new_user_ID = 0

new_user_ratings = [
     (0, 260, 4),   # Star Wars (1977)
     (0, 1, 3),     # Toy Story (1995)
     (0, 16, 3),    # Casino (1995)
     (0, 25, 4),    # Leaving Las Vegas (1995)
     (0, 32, 4),    # Twelve Monkeys (1995)
     (0, 335, 1),   # The Flintstones (1994)
     (0, 379, 1),   # Timecop (1994)
     (0, 296, 3),   # Pulp Fiction (1994)
     (0, 858, 5),   # The Godfather (1972)
     (0, 50, 4)     # The Usual Suspects (1995)
]

new_user_ratings_RDD = sc.parallelize(new_user_ratings)

print(f'New user ratings: {new_user_ratings_RDD.take(10)}')


New user ratings: [(0, 260, 4), (0, 1, 3), (0, 16, 3), (0, 25, 4), (0, 32, 4), (0, 335, 1), (0, 379, 1), (0, 296, 3), (0, 858, 5), (0, 50, 4)]


In [22]:
complete_data_with_new_ratings_RDD = complete_ratings_data.union(new_user_ratings_RDD)


In [24]:
from time import time

t0 = time()
new_ratings_model = ALS.train(complete_data_with_new_ratings_RDD, best_rank, seed=seed, 
                              iterations=iterations, lambda_=regularization_parameter)
tt = time() - t0

print(f"New model trained in {round(tt, 3)} seconds")


Py4JJavaError: An error occurred while calling o672.trainALSModel.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 1105.0 failed 1 times, most recent failure: Lost task 0.0 in stage 1105.0 (TID 1417, localhost, executor driver): java.io.IOException: No space left on device
	at java.io.FileOutputStream.writeBytes(Native Method)
	at java.io.FileOutputStream.write(FileOutputStream.java:326)
	at org.apache.spark.storage.TimeTrackingOutputStream.write(TimeTrackingOutputStream.java:58)
	at java.io.BufferedOutputStream.flushBuffer(BufferedOutputStream.java:82)
	at java.io.BufferedOutputStream.write(BufferedOutputStream.java:126)
	at net.jpountz.lz4.LZ4BlockOutputStream.flushBufferedData(LZ4BlockOutputStream.java:220)
	at net.jpountz.lz4.LZ4BlockOutputStream.write(LZ4BlockOutputStream.java:173)
	at java.io.ObjectOutputStream$BlockDataOutputStream.drain(ObjectOutputStream.java:1877)
	at java.io.ObjectOutputStream$BlockDataOutputStream.writeByte(ObjectOutputS

### getting top recommendations

In [20]:
# Get just the movie IDs rated by the new user
new_user_ratings_ids = list(map(lambda x: x[1], new_user_ratings))

# Create RDD with movies the new user hasn't rated yet
new_user_unrated_movies_RDD = complete_movies_data \
    .filter(lambda x: x[0] not in new_user_ratings_ids) \
    .map(lambda x: (new_user_ID, x[0]))

# Predict ratings for the new user's unrated movies
new_user_recommendations_RDD = new_ratings_model.predictAll(new_user_unrated_movies_RDD)


NameError: name 'new_user_ratings' is not defined

In [25]:
new_user_recommendations_rating_RDD = new_user_recommendations_RDD.map(lambda x: (x.product, x.rating))
recommendations_with_titles_RDD = new_user_recommendations_rating_RDD.join(complete_movies_titles)
new_user_recommendations_rating_title_and_count_RDD = recommendations_with_titles_RDD.join(movie_rating_counts_RDD)
new_user_recommendations_rating_title_and_count_RDD.take(3)


[(257805, ((2.5085718670342345, 'Best Sellers (2021)'), 32)),
 (154530, ((0.7315767397154207, 'Recto / verso'), 2)),
 (71910, ((1.439479763089496, '"Tournament'), 268))]

In [26]:
new_user_recommendations_rating_title_and_count_RDD = new_user_recommendations_rating_title_and_count_RDD.map(
    lambda r: (r[1][0][1], r[1][0][0], r[1][1])
)


In [27]:
top_movies = new_user_recommendations_rating_title_and_count_RDD \
    .filter(lambda r: r[2] >= 25) \
    .takeOrdered(25, key=lambda x: -x[1])

print('TOP recommended movies (with more than 25 reviews):\n%s' %
      '\n'.join(map(str, top_movies)))


TOP recommended movies (with more than 25 reviews):
('Mugabe and the White African (2009)', 4.282686215571786, 27)
('"Lewis Black: Red', 4.156715066700028, 25)
('"Human Condition III', 4.121946089062461, 145)
('"Last Lions', 4.087593714503657, 51)
('The Web (2013)', 4.058388142488241, 26)
('High School (1968)', 4.044746741656282, 57)
('Harakiri (Seppuku) (1962)', 4.0123147644975194, 1282)
("Long Night's Journey Into Day (2000)", 4.006672444468443, 36)
('DamNation (2014)', 3.9991908273946404, 26)
('Wow! A Talking Fish! (1983)', 3.9957715303424566, 90)
('"Mystery of Picasso', 3.982514338395071, 32)
('Heimat - A Chronicle of Germany (Heimat - Eine deutsche Chronik) (1984)', 3.9782987072965934, 37)
('Rabbit of Seville (1950)', 3.955434901330867, 259)
('Martha (1974)', 3.9504535570946953, 26)
('Norm MacDonald: Me Doing Standup (2011)', 3.942519554648639, 36)
('"Organizer', 3.942319728676967, 29)
('Streetwise (1984)', 3.928272760695844, 39)
('"Great War', 3.927808211986288, 39)
('Seven Samur

### getting individual ratings

In [28]:
my_movie = sc.parallelize([(0, 500)])  # (userID, movieID)
individual_movie_rating_RDD = new_ratings_model.predictAll(my_movie)
print(individual_movie_rating_RDD.take(1))


[Rating(user=0, product=500, rating=2.073246660223336)]


## Final Project


In [12]:
from pyspark.mllib.recommendation import ALS
import math
from time import time


In [13]:
# Define user IDs
user1_ID = -1
user2_ID = -2

# Define ratings for each user
user1_ratings = [
    (user1_ID, 1, 4), (user1_ID, 50, 5), (user1_ID, 260, 5),
    (user1_ID, 296, 4), (user1_ID, 318, 4), (user1_ID, 527, 2),
    (user1_ID, 589, 3), (user1_ID, 858, 5), (user1_ID, 1196, 5), (user1_ID, 1210, 4)
]

user2_ratings = [
    (user2_ID, 1, 2), (user2_ID, 110, 4), (user2_ID, 150, 5), (user2_ID, 165, 4),
    (user2_ID, 260, 3), (user2_ID, 276, 5), (user2_ID, 329, 1), (user2_ID, 527, 4),
    (user2_ID, 1100, 5), (user2_ID, 1214, 4)
]


In [14]:
# # Load and clean the ratings dataset
# complete_ratings_file = os.path.join(datasets_path, 'ml-latest', 'ratings.csv')
# complete_ratings_raw_data = sc.textFile(complete_ratings_file)
# complete_ratings_raw_data_header = complete_ratings_raw_data.take(1)[0]

# complete_ratings_data = complete_ratings_raw_data \
#     .filter(lambda line: line != complete_ratings_raw_data_header) \
#     .map(lambda line: line.split(",")) \
#     .map(lambda tokens: (int(tokens[0]), int(tokens[1]), float(tokens[2]))) \
#     .cache()

# # Clean ratings (remove bad types or NaNs)
# complete_ratings_data_clean = complete_ratings_data \
#     .filter(lambda x: x is not None and len(x) == 3 and
#             isinstance(x[0], int) and isinstance(x[1], int) and isinstance(x[2], float) and
#             x[2] == x[2])  # removes NaN

# # Load the movies dataset
# complete_movies_file = os.path.join(datasets_path, 'ml-latest', 'movies.csv')
# complete_movies_raw_data = sc.textFile(complete_movies_file)
# complete_movies_raw_data_header = complete_movies_raw_data.take(1)[0]

# complete_movies_data = complete_movies_raw_data \
#     .filter(lambda line: line != complete_movies_raw_data_header) \
#     .map(lambda line: line.split(",")) \
#     .map(lambda tokens: (int(tokens[0]), tokens[1], tokens[2])) \
#     .cache()

# # Extract movie titles (MovieID, Title)
# complete_movies_titles = complete_movies_data.map(lambda x: (int(x[0]), x[1]))
# Load and clean the ratings dataset
complete_ratings_file = os.path.join(datasets_path, 'ml-latest', 'ratings.csv')
complete_ratings_raw_data = sc.textFile(complete_ratings_file)
complete_ratings_raw_data_header = complete_ratings_raw_data.take(1)[0]

complete_ratings_data = complete_ratings_raw_data \
    .filter(lambda line: line != complete_ratings_raw_data_header) \
    .map(lambda line: line.split(",")) \
    .map(lambda tokens: (int(tokens[0]), int(tokens[1]), float(tokens[2]))) \
    .cache()

# Clean it
complete_ratings_data_clean = complete_ratings_data \
    .filter(lambda x: x is not None and len(x) == 3 and
            isinstance(x[0], int) and isinstance(x[1], int) and isinstance(x[2], float) and
            x[2] == x[2])  # removes NaNs


In [15]:
movie_ID_with_ratings_RDD = complete_ratings_data.map(lambda x: (x[1], x[2])).groupByKey()

movie_ID_with_avg_ratings_RDD = movie_ID_with_ratings_RDD.map(
    lambda x: (x[0], (len(x[1]), float(sum(x[1])) / len(x[1])))
)

movie_rating_counts_RDD = movie_ID_with_avg_ratings_RDD.map(lambda x: (x[0], x[1][0]))


In [16]:
user1_ratings_RDD = sc.parallelize(user1_ratings)
data_with_user1 = complete_ratings_data_clean.union(user1_ratings_RDD).map(lambda x: (int(x[0]), int(x[1]), float(x[2])))



In [17]:
model1 = ALS.train(data_with_user1, best_rank, seed=seed, iterations=iterations, lambda_=regularization_parameter)

In [19]:
# Load the movies dataset again
complete_movies_file = os.path.join(datasets_path, 'ml-latest', 'movies.csv')
complete_movies_raw_data = sc.textFile(complete_movies_file)
complete_movies_raw_data_header = complete_movies_raw_data.take(1)[0]

complete_movies_data = complete_movies_raw_data \
    .filter(lambda line: line != complete_movies_raw_data_header) \
    .map(lambda line: line.split(",")) \
    .map(lambda tokens: (int(tokens[0]), tokens[1], tokens[2])) \
    .cache()

complete_movies_titles = complete_movies_data.map(lambda x: (int(x[0]), x[1]))


In [20]:
user1_rated_ids = set([x[1] for x in user1_ratings])
user1_unrated_RDD = complete_movies_data.filter(lambda x: x[0] not in user1_rated_ids).map(lambda x: (user1_ID, x[0]))
user1_predictions = model1.predictAll(user1_unrated_RDD).map(lambda r: (r.product, r.rating))
user1_joined = user1_predictions.join(complete_movies_titles).join(movie_rating_counts_RDD)

In [21]:
# Scenario 1 – min 25 ratings
print("\nUser 1 – Scenario 1 (min 25 ratings)")
user1_s1 = user1_joined.filter(lambda r: r[1][1] >= 25).map(
    lambda r: (r[1][0][1], r[1][0][0], r[1][1])  # (title, predicted_rating, count)
)
top15_user1_s1 = user1_s1.takeOrdered(15, key=lambda x: -x[1])
for movie in top15_user1_s1:
    print(movie)



User 1 – Scenario 1 (min 25 ratings)
('DamNation (2014)', 4.795507355625961, 26)
('Assassins (2020)', 4.688805316746993, 33)
('World of Glory (1991)', 4.55630896634093, 33)
('Attack on Titan: Chronicle (2020)', 4.550009492195567, 29)
('Kizumonogatari II: Passionate Blood (2016)', 4.543482647854702, 71)
('"Lord of the Rings: The Fellowship of the Ring', 4.490473855874679, 79940)
('"Lord of the Rings: The Return of the King', 4.478225344071891, 75512)
('"Lord of the Rings: The Two Towers', 4.446505299149898, 73687)
('"Come Sweet Death (Komm', 4.444727880865571, 31)
('Little Fugitive (1953)', 4.443216624134763, 27)
('Garbage Warrior (2007)', 4.435742727753592, 27)
('Ghost in the Shell Arise - Border 5: Pyrophoric Cult (2015)', 4.3787799983054825, 55)
('Spider-Man: Across the Spider-Verse (2023)', 4.36683437760049, 528)
('The Work (2017)', 4.334267755549121, 37)
('Mushishi: The Shadow That Devours the Sun (2014)', 4.328509826573175, 33)


In [33]:
# Scenario 2 – min 100 ratings
print("\nUser 1 – Scenario 2 (min 100 ratings)")
user1_s2 = user1_joined.filter(lambda r: r[1][1] >= 100).map(
    lambda r: (r[1][0][1], r[1][0][0], r[1][1])
)
top15_user1_s2 = user1_s2.takeOrdered(15, key=lambda x: -x[1])
for movie in top15_user1_s2:
    print(movie)



User 1 – Scenario 2 (min 100 ratings)
('"Lord of the Rings: The Fellowship of the Ring', 4.490473855874679, 79940)
('"Lord of the Rings: The Return of the King', 4.478225344071891, 75512)
('"Lord of the Rings: The Two Towers', 4.446505299149898, 73687)
('Spider-Man: Across the Spider-Verse (2023)', 4.36683437760049, 528)
('Cosmos', 4.303691007402421, 625)
('Planet Earth II (2016)', 4.283464427642407, 2041)
('In This Corner of the World (2016)', 4.273790693720311, 154)
('Seven Samurai (Shichinin no samurai) (1954)', 4.272358115560259, 17120)
('Band of Brothers (2001)', 4.2685753258215655, 2835)
('Ghost in the Shell: Stand Alone Complex - The Laughing Man (2005)', 4.250545698596042, 383)
('Planet Earth (2006)', 4.216997341518166, 3015)
('Ghost in the Shell Arise - Border 3: Ghost Tears (2014)', 4.204562618254329, 114)
('Kill Bill: The Whole Bloody Affair (2011)', 4.197659530240273, 241)
('Yojimbo (1961)', 4.197016475495417, 5208)
('"Godfather: Part II', 4.192121023213467, 47271)


In [16]:
user2_ratings_RDD = sc.parallelize(user2_ratings)
data_with_user2 = complete_ratings_data_clean.union(user2_ratings_RDD).map(lambda x: (int(x[0]), int(x[1]), float(x[2])))



In [17]:
complete_ratings_data_clean = sc.textFile(os.path.join(datasets_path, 'ml-latest', 'ratings.csv')) \
    .filter(lambda line: not line.startswith("userId")) \
    .map(lambda line: line.split(",")) \
    .map(lambda tokens: (int(tokens[0]), int(tokens[1]), float(tokens[2]))) \
    .filter(lambda x: x is not None and x[2] == x[2]) \
    .cache()

In [18]:
complete_movies_file = os.path.join(datasets_path, 'ml-latest', 'movies.csv')
complete_movies_raw_data = sc.textFile(complete_movies_file)
complete_movies_raw_data_header = complete_movies_raw_data.take(1)[0]

complete_movies_data = complete_movies_raw_data \
    .filter(lambda line: line != complete_movies_raw_data_header) \
    .map(lambda line: line.split(",")) \
    .map(lambda tokens: (int(tokens[0]), tokens[1], tokens[2])) \
    .cache()

complete_movies_titles = complete_movies_data.map(lambda x: (int(x[0]), x[1]))


In [19]:
model2 = ALS.train(data_with_user2, best_rank, seed=seed, iterations=iterations, lambda_=regularization_parameter)


In [20]:
user2_rated_ids = set([x[1] for x in user2_ratings])
user2_unrated_RDD = complete_movies_data.filter(lambda x: x[0] not in user2_rated_ids).map(lambda x: (user2_ID, x[0]))
user2_predictions = model2.predictAll(user2_unrated_RDD).map(lambda r: (r.product, r.rating))
user2_joined = user2_predictions.join(complete_movies_titles).join(movie_rating_counts_RDD)


In [22]:
# Scenario 1 – min 25 ratings
print("\nUser 2 – Scenario 1 (min 25 ratings)")
user2_s1 = user2_joined.filter(lambda r: r[1][1] >= 25).map(
    lambda r: (r[1][0][1], r[1][0][0], r[1][1])  # (title, predicted_rating, count)
)
top15_user2_s1 = user2_s1.takeOrdered(15, key=lambda x: -x[1])
for movie in top15_user2_s1:
    print(movie)



User 2 – Scenario 1 (min 25 ratings)
('What a Beautiful Day (2011)', 5.194521886058748, 31)
('Like Minds (2006)', 5.0427670596289165, 35)
('Recep İvedik 4 (2014)', 5.021113608523292, 27)
('Cado dalle Nubi (2009)', 5.007903938555204, 26)
('"Naked Man', 4.996377923969636, 28)
('Katt Williams: The Pimp Chronicles Pt. 1 (2006)', 4.94545299235361, 30)
('Shaadi Mein Zaroor Aana (2017)', 4.936939319246031, 26)
('Amy Schumer: Mostly Sex Stuff (2012)', 4.920470645969265, 31)
("Kevin Hart: I'm a Grown Little Man (2009)", 4.912681509401093, 91)
("Jeff Dunham: Jeff Dunham's Very Special Christmas Special (2008)", 4.911810422542893, 26)
('The Chaos Class Failed the Class (1976)', 4.895194077403689, 37)
('Dil Dhadakne Do (2015)', 4.891871562143466, 66)
('Recep İvedik 3 (2010)', 4.890275786036936, 28)
('The Cyborgs (2017)', 4.868174181918891, 27)
('Food Matters (2008)', 4.855884256565362, 42)


In [23]:
 # Scenario 2 – min 100 ratings
print("\nUser 2 – Scenario 2 (min 100 ratings)")
user2_s2 = user2_joined.filter(lambda r: r[1][1] >= 100).map(
    lambda r: (r[1][0][1], r[1][0][0], r[1][1])
)
top15_user2_s2 = user2_s2.takeOrdered(15, key=lambda x: -x[1])
for movie in top15_user2_s2:
    print(movie)



User 2 – Scenario 2 (min 100 ratings)
('My Sassy Girl (2008)', 4.838600980239347, 172)
('Prison Break: The Final Break (2009)', 4.81295188213145, 304)
('Live Nude Girls (1995)', 4.803864693865478, 817)
('Now You See Me (2013)', 4.781055359371335, 11029)
('"Life of David Gale', 4.772481317829096, 3191)
('Peaceful Warrior (2006)', 4.762518446903648, 446)
('Law Abiding Citizen (2009)', 4.74229567472395, 4192)
('13 reasons why', 4.736772169364379, 381)
('Now You See Me 2 (2016)', 4.732114765899945, 4697)
('Five Feet Apart (2019)', 4.7309451821365816, 365)
('Talking About Sex (1994)', 4.719437187224074, 118)
('Missing (2023)', 4.71518737339068, 116)
('"Love', 4.713854992125585, 1119)
('Zeitgeist: Addendum (2008)', 4.712440976112768, 438)
('Kidnap (2017)', 4.702760341517159, 143)
